In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
project_path = "/content/drive/MyDrive/Colab Notebooks/fathom_financial_agent"

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-yec_rr2d/unsloth_0f8f628e7cb140b2ab9af9931f719c5b
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-yec_rr2d/unsloth_0f8f628e7cb140b2ab9af9931f719c5b
  Resolved https://github.com/unslothai/unsloth.git to commit ab4061e106792fa91e1eba3e4f3d45fa8aba121e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import json
import pandas as pd
from unsloth import FastLanguageModel
from tqdm import tqdm
import torch
import ast

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
max_seq_length = 4096
dtype = None
load_in_4bit = True

In [ ]:
def run_inference(model, tokenizer, data, desc):
    results = []
    print(f"--- Running {desc} ---")

    system_prompt = """You are a financial analyst. Answer the user's question based on the context provided. Respond with exactly one <reasoning> block and one <answer> block."""

    for entry in tqdm(data, desc=desc):
        if "input" in entry:
            user_content = entry["input"]
            gold_answer = entry.get("output", "")
            doc_id = entry.get("metadata", {}).get("financebench_id", "unknown")
        else:
            raw_evidence = entry.get("evidence", "")
            context_text = raw_evidence
            if isinstance(raw_evidence, str) and raw_evidence.strip().startswith("["):
                parsed_evidence = ast.literal_eval(raw_evidence)
                context_text = "\n\n".join([item.get("evidence_text", "") for item in parsed_evidence])

            question = entry.get("question", "")
            user_content = f"Context:\n{context_text}\n\nQuestion:\n{question}"
            gold_answer = entry.get("answer", "")
            doc_id = entry.get("financebench_id", "unknown")

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ]

        prompt = tokenizer.apply_chat_template(
          messages,
          tokenize=False,
          add_generation_prompt=True,
        )

        inputs = tokenizer(
          prompt,
          return_tensors="pt",
          padding=False,
        )
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
            temperature=0.1,
            repetition_penalty=1.2
        )

        response = tokenizer.batch_decode(outputs)
        text = response[0].split("<|start_header_id|>assistant<|end_header_id|>\n\n")[-1].replace("<|eot_id|>", "")

        results.append({
            "id": doc_id,
            "question": entry.get("question", "Unknown"),
            "teacher_answer": gold_answer,
            "pred_answer": text
        })
    return results

In [ ]:
data = []
with open(os.path.join(project_path, "test.jsonl"), 'r') as f:
    for line in f:
        data.append(json.loads(line))
print(f"Loaded {len(data)} Hold-out examples.")

Loaded 15 Hold-out examples.


In [ ]:
print("\nLoading BASE Llama-3.2-3B-Instruct...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model)

base_results = run_inference(model, tokenizer, data, "Base Model")
pd.DataFrame(base_results).to_json(f"{project_path}/baseline_holdout_results.json", orient="records", indent=2)


Loading BASE Llama-3.2-3B-Instruct...
==((====))==  Unsloth 2026.1.3: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
--- Running Base Model ---


Base Model: 100%|██████████| 15/15 [02:10<00:00,  8.70s/it]


In [ ]:
del model
torch.cuda.empty_cache()

In [ ]:
print("\nLoading FINE-TUNED Adapters...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


Loading FINE-TUNED Adapters...
==((====))==  Unsloth 2026.1.3: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
adapter_path = "qlora_adapters"
if not os.path.exists(os.path.join(project_path, adapter_path)):
    print("Path wasn't found.")

In [ ]:
print(f"Loading adapters from: {os.path.join(project_path, adapter_path)}")
model.load_adapter(os.path.join(project_path, adapter_path))
FastLanguageModel.for_inference(model)

Loading adapters from: /content/drive/MyDrive/Colab Notebooks/fathom_financial_agent/qlora_adapters


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=3072, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=3072, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=3072, out_features=1024, bias=False)
            (lora_dropout): ModuleDict(
              

In [ ]:
ft_results = run_inference(model, tokenizer, data, "Fine-Tuned Model")
pd.DataFrame(ft_results).to_json(f"{project_path}/finetuned_holdout_results.json", orient="records", indent=2)

print(f"\nResults saved to {project_path}/")

--- Running Fine-Tuned Model ---


Fine-Tuned Model: 100%|██████████| 15/15 [04:00<00:00, 16.01s/it]


Phase 2 Complete. Results saved to /content/drive/MyDrive/Colab Notebooks/fathom_financial_agent/
